In [ ]:
# This Jupyter Notebook loads and runs various simulations and analyses concerning
# the planaria model reported on in the manuscript, "Neural Control of Body-plan 
# Axis in Regenerating Planaria" by Pietak et al. submitted to PLoS Comp Bio.  

# To use this notebook with the Planaria Interface for Modeling Body Organization 
# (PLIMBO) simulator, please first follow the installation instructions for the 
# BETSE simulator to install necessary dependencies:

# https://gitlab.com/betse/betse

# And then download the PLIMBO project:

# https://gitlab.com/betse/plimbo

# 

# The PLIMBO project folder contains all accessory configuration files and other data 
# required to run planaria model simulations. You can find saved data from 
# simulations that you run in this Notebook (e.g. plots and 
# animations) in the PLIMBO/plimbo/data folder on your system.

In [ ]:
# Import statements
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from matplotlib import colors 
from matplotlib import colorbar
from collections import OrderedDict
from scipy.ndimage import rotate
import copy
import pickle
import os
import os.path
import sys, time
import csv
from betse.lib.pickle import pickles
from matplotlib import rcParams

from plimbo.plimbo import PlimboRunner
from plimbo.sim2D import PlanariaGRN2D

from matplotlib.collections import PolyCollection
from betse.science.math import toolbox as tb

# --- SOVEREIGN REPLACEMENT FOR DEPRECATED SCIPY IMRESIZE ---
try:
    from betse.util.serial.pickle import pickles
except ImportError:
    from betse.lib.pickle import pickles

from PIL import Image

def imresize(arr, size):
    """Sovereign fallback function for modern SciPy environments"""
    # Cast to uint8 image array
    img = Image.fromarray((arr * 255).astype(np.uint8))
    if isinstance(size, (float, int)):
        new_size = (int(img.width * size), int(img.height * size))
    else:
        # Swap dimension positions because legacy SciPy used (H, W) 
        # while Pillow natively expects (W, H)
        new_size = (size[1], size[0])
    return np.array(img.resize(new_size, resample=Image.Resampling.BILINEAR))

In [ ]:
# Parameters set-up:

# Finalized parameters for the planaria reaction-convection-diffusion model
# with Markov model parameters included (these parameter values can be edited, but don't
# change the order of the parameters listed):

# All parameters are defined in full (with units specified) in the 
# Supporting Information S1 document. 

param_o = OrderedDict({ 
            'r_bc': 2.5e-3,  #
            'd_bc': 1.0e-6,  #
            'd_bc_deg': 5.0e-3,  #
            'K_bc_apc': 0.5,
            'n_bc_apc': 2.0,
            'K_bc_camp': 1.0,
            'n_bc_camp': 2.0,
            'D_bc': 1.0e-12,

            # ERK parameters
            'K_erk_bc': 30.0,  #
            'n_erk_bc': 2.0,  #

            # APC parameters
            'K_apc_wnt': 30.0,  #
            'n_apc_wnt': 2.0,  #

            # Hedgehog parameters:
            'r_hh': 5.0e-3,  #
            'd_hh': 5.0e-6,  #
            'D_hh': 1.5e-11,  #
            'u_hh': 3.0e-8,  #

            # Wnt1 parameters
            'r_wnt': 1.0e-2,  #
            'd_wnt': 1.0e-5,  #
            'K_wnt_notum': 0.5,  #
            'n_wnt_notum': 2.5,  # 2.5
            'D_wnt': 5.0e-12,  #
            'd_wnt_deg_notum': 1.0e-4,  # 1.0e-4
            'd_wnt_deg_ptc': 7.5e-5,  #
            'K_wnt_hh': 300.0,  #
            'n_wnt_hh': 2.5,  # 2.5

            # NRF parameters
            'r_nrf': 7.5e-3,  #
            'd_nrf': 7.5e-6,  #
            'K_nrf_bc': 150.0,  # 150
            'n_nrf_bc': 2.0,  # 2.0
            'D_nrf': 1.5e-11,  #
            'u_nrf': -6.0e-8,  #

            # Notum parameters
            'r_notum': 2.5e-3,  #
            'd_notum': 2.5e-3,  #
            'K_notum_nrf': 250.0,  # 250, 150
            'n_notum_nrf': 2.0,  # 2.0, 1.0
            'D_notum': 1.5e-11,  #

            # Markov model parameters:
            'C1': 0.75,  # ERK constant to modulate head formation
            'K1': 0.05,

            'C2': 300.0,  # Beta-catenin concentration to modulate tail formation
            'K2': 40.0,

            'Beta_B': 5.0e-3,  # head/tail tissue decay time constant

            'hdac_to': 90.0 * 3600,  # 96 time at which hdac stops growing
            'hdac_ts': 12.0 * 3600,  # time period over which hdac stops growing

            'nd_min': 0.4, # minimum value of the nerve map
            'nd_max': 1.0  # maximu value of the nerve map
})

# Note that small modifications are required to work with 2H worms -- therefore copy the above
# base dictionary, and use this separate set of parameters for 2H models.
param_2H = param_o.copy()
param_2H['K_erk_bc'] = 50.0  # can be as high as 75
param_2H['nd_max'] = 2.0 # stretch the contrast of the nerve density map

blnkRNAi = OrderedDict() # empty dictionary for omitting RNAi test sequence in models

In [ ]:
# MODEL ACCESS and DATA SAVING
#-----------------------------
# For simplicity, the default 1H worm simulation (4 cuts) will be used here:
model_key = '1HWorm_Default'

# Using the default 1H worm model, data produced by simulations run
# in this Notebook will be saved to (relative to PLIMBO's location on your system):
# e.g. 'PLIMBO/plimbo/data/BasicTesting/1H1/'

# All available model keys can be printed; use one of these to select 
# a different worm model (e.g. model_key = '2HWorm1_1Cut_sym', with head_frags = [0] and
# tail_frags = [1]):

sim = PlimboRunner(head_frags = [0], tail_frags = [4]) 
sim.print_available_models()

# Note that if you want to run, edit the configuration and save a model to a different 
# directory on your system, first copy a whole model sub-folder from the 
# 'PLIMBO/plimbo/data directory' (e.g. copy the entire '1H1Worm' folder) to a new
# location on your system, and then supply a path to the location of the yaml
# file. An an example, if the new yaml is located at: 
# my_config = '/home//Planaria_Project/1H1Worm/1H_ellipse.yaml'
# Then change all calls to the simulator to:
# sim = PlimboRunner(fn_config = my_config, head_frags = [0], tail_frags = [4]) 
# When fn_config is changed from fn_config = None, it will override the worm_model
# key and try to read the model from your specified path. All data will be saved relative
# to where the model yaml config file is on your system.

# Also, head and tail fragments are ordered from the head (0), and the index
# changes depending on how many fragments the worm is cut into. For example, if there 
# are 5 fragments (4 cuts), then the head fragment is indexed 0 and the tail at 4, 
# but if there are 2 fragments (1 cut), then the head fragment is indexed 0 and the tail 1.
# These fragment indices for head and tail must be specified by the user at the time of
# cutting.

In [ ]:
## The following code blocks can be commented in to run different types of simulations.

## Note that available plot types are:
## 'Triplot', 'Biplot', 'Hexplot' and 'Markovplot' (you may need to adjust 'fsize' for 
## plot type and for different worm models)
## Note available animation types are:
## 'Triplot' and 'Biplot' (you may need to adjust fsize for
## plot type and for different worm models)

In [ ]:
# # Run a single body-model simulation in 1D without any RNAi/intervention testing:

# sim = PlimboRunner(worm_model = model_key, fn_config = None,
#                    head_frags = [0], tail_frags = [4]) 
# sim.simRNAi(RNAi_vect = blnkRNAi, RNAi_tags = blnkRNAi, params = param_o, run_time_init = 3600*168,
#                 run_time_sim = 3600*168, run_time_step = 60, run_time_sample = 25.0,
#                 run_time_reinit = 0.0,  xscale = 1.0, verbose = True, new_mesh = False,
#                 save_dir = 'sim_1D', reset_clims = True, animate = False, plot = True,
#                 plot_type = ['Markovplot','Hexplot'], ani_type = 'Triplot', save_harness = False,
#                 harness_type='1D', fsize = [(12,12), (18,12)])

In [ ]:
# # Run a single body-model simulation in 2D without any RNAi/intervention testing:
# sim = PlimboRunner(worm_model = model_key, fn_config = None,
#       head_frags = [0], tail_frags = [4]) 
# sim.simRNAi(RNAi_vect = blnkRNAi, RNAi_tags = blnkRNAi, params = param_o, run_time_init = 3600*168,
#                 run_time_sim = 3600*168, run_time_step = 60, run_time_sample = 25.0,
#                 run_time_reinit = 0.0,  xscale = 1.0, verbose = True, new_mesh = False,
#                 save_dir = 'sim_2D', reset_clims = False, animate = False, plot = True,
#                 plot_type = ['Markovplot','Hexplot'], ani_type = 'Triplot', save_harness = False,
#                 harness_type='2D', fsize = [(6,8),(6,14)])

In [ ]:
# # Run a single body-model simulation with RNAi/intervetions test sequence in 1D:
# sim = PlimboRunner(worm_model = model_key, fn_config = None,
#        head_frags = [0], tail_frags = [4]) 
# sim.simRNAi(params = param_o, run_time_init = 3600*108,
#                 run_time_sim = 3600*108, run_time_step = 60.0, run_time_sample = 50.0,
#                 run_time_reinit = 0.0,  xscale = 1.0, verbose = True, new_mesh = False,
#                 save_dir = 'SimRNAi_1D', reset_clims = True, animate = False, plot = True,
#                 plot_type = ['Markovplot','Hexplot'], ani_type = 'Triplot', save_harness = False,
#                 harness_type='1D', fsize = [(12,12), (18,12)])

In [ ]:
# # Run a single body-model simulation in 2D with RNAi/intervention testing:
# sim = PlimboRunner(worm_model = model_key, fn_config = None,
#                   head_frags = [0], tail_frags = [4]) 
# sim.simRNAi(params = param_o, run_time_init = 3600*96,
#                 run_time_sim = 3600*96, run_time_step = 60, run_time_sample = 50.0,
#                 run_time_reinit = 0.0,  xscale = 1.0, verbose = True, new_mesh = False,
#                 save_dir = 'simRNAi_2D', reset_clims = False, animate = False, plot = True,
#                 plot_type = ['Markovplot','Hexplot'], ani_type = 'Triplot', save_harness = False,
#                 harness_type='2D', fsize = [(6,8),(6,14)])

In [ ]:
# # Run a single body-model simulation with full RNAi tests in 2D with 7 day
# # 'reinitialization' phase:
# sim = PlimboRunner(worm_model = model_key, fn_config = None,
#                    head_frags = [0], tail_frags = [4]) 
# sim.simRNAi(params = param_o, run_time_init = 3600*96,
#                 run_time_sim = 3600*96, run_time_step = 60.0, run_time_sample = 50.0,
#                 run_time_reinit = 3600.0*168,  xscale = 1.0, verbose = True, new_mesh = False,
#                 save_dir = 'SimRNAi_2D_longreinit', reset_clims = False, animate = True, plot = True,
#                 plot_type = ['Markovplot','Hexplot'], ani_type = 'Triplot', save_harness = False,
#                 harness_type='2D', fsize = [(6,8),(6,14)])

In [ ]:
# # Run a sensitivity analysis on the model in 1D:
# sim = PlimboRunner(worm_model = model_key, fn_config = None,
#                   head_frags = [0], tail_frags = [4])
# sim.sensitivity(params=param_o, 
#                 run_type = 'sim', 
#                 run_time_init=3600*96, 
#                 factor = 0.15,
#                 run_time_sim=3600*96, 
#                 run_time_step=60.0, 
#                 run_time_sample=50.0,
#                  xscale=1.0, 
#                 verbose=True, 
#                 new_mesh=False, 
#                 ani_type = 'Triplot',
#                 save_dir='Sensitivity_1D', 
#                 reset_clims=True, 
#                 animate=False, plot=True,
#                 plot_type=['Markovplot','Hexplot'],
#                 save_harness=False, 
#                 harness_type='1D',
#                 reference = ['Head','Tail','Erk','β-Cat','Notum','Hh'], 
#                 fsize = [(12,12), (18,12)],
#                 paramo_units = None)

In [ ]:
# # Run a sensitivity analysis on the model in 2D:
# sim = PlimboRunner(worm_model = model_key, fn_config = None,
#                    head_frags = [0], tail_frags = [4])
# sim.sensitivity(params=param_o, 
#                 run_type = 'sim', 
#                 run_time_init=3600*96, 
#                 factor = 0.15,
#                 run_time_sim=3600*96, 
#                 run_time_step=60.0, 
#                 run_time_sample=50.0,
#                  xscale=1.0, 
#                 verbose=True, 
#                 new_mesh=False, 
#                 ani_type = 'Triplot',
#                 save_dir='Sensitivity_2D', 
#                 reset_clims=True, 
#                 animate=False, 
#                 plot=True,
#                 plot_type=['Markovplot','Hexplot'], 
#                 save_harness=False, 
#                 harness_type='2D',
#                reference = ['Head','Tail','Erk','β-Cat','Notum','Hh'], 
#                 fsize = [(6,8),(6,14)], 
#                 axisoff = True, 
#                 )

In [ ]:
# # Run a local parameter search for model improvements in 1D:
# sim = PlimboRunner(worm_model = model_key, fn_config = None,
#                   head_frags = [0], tail_frags = [4]) 
# sim.searchRNAi(params=param_o, free_params = None, run_time_init = 3600.0*96,
#                 run_time_sim = 3600.0*96, run_time_step = 90.0, run_time_sample = 50.0, search_style = 'log',
#                    factor = 0.66, levels = 1, run_time_reinit = 0.0, xscale = 1.0, verbose = True, new_mesh = False,
#                 save_dir = 'SearchRNAi_1D', reset_clims = True, animate = False, plot = True, ani_type = 'Triplot',
#                 plot_type = ['Markovplot','Hexplot'], save_harness = False, harness_type='1D',
#                   fsize = [(12,12), (18,12)])

In [ ]:
# # Run a local parameter search for model improvements in 2D:
# sim = PlimboRunner(worm_model = model_key, fn_config = None,
#                     head_frags = [0], tail_frags = [4]) 
# sim.searchRNAi(params=param_o, 
#                free_params = None, 
#                xscale = 1.0, 
#                harness_type='2D',
#                save_dir = 'SearchRNAi_2D', 
#                run_time_init = 3600.0*96, 
#                run_time_reinit = 0.0, 
#                run_time_sim = 3600.0*96, 
#                run_time_step = 90.0, 
#                run_time_sample = 50.0, 
#                search_style = 'log',
#                factor = 0.75, 
#                levels = 2, 
#                verbose = True, 
#                new_mesh = False,
#                reset_clims = True, 
#                animate = False, 
#                plot = True, 
#                ani_type = 'Triplot',
#                plot_type = ['Markovplot','Hexplot'],
#                save_harness = False, 
#                axisoff = False,
#                fsize = [(6,8),(6,14)],
#                up_only = False)